# Sitzung 2: Kurvendiskussion — Interaktiv

Hier gehst du Schritt für Schritt eine komplette Kurvendiskussion durch und **siehst** dabei, was jeder Schritt bedeutet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, Dropdown, Checkbox
from IPython.display import display, Markdown
import sympy as sp

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['mathtext.fontset'] = 'cm'

---
## 1. f, f' und f'' im Zusammenhang

Wähle eine Funktion und blende f'(x) und f''(x) ein/aus. Beobachte den Zusammenhang!

In [ ]:
x_sym = sp.Symbol('x')

funktionen = {
    'x³ − 3x': x_sym**3 - 3*x_sym,
    'x⁴ − 4x² + 1': x_sym**4 - 4*x_sym**2 + 1,
    '-x³ + 6x² − 9x': -x_sym**3 + 6*x_sym**2 - 9*x_sym,
    'x · e⁻ˣ': x_sym * sp.exp(-x_sym),
    'sin(x)': sp.sin(x_sym),
}

def zusammenhang(wahl='x³ − 3x', zeige_f_prime=True, zeige_f_double_prime=True):
    f_sym = funktionen[wahl]
    fp_sym = sp.diff(f_sym, x_sym)
    fpp_sym = sp.diff(fp_sym, x_sym)
    
    f_num = sp.lambdify(x_sym, f_sym, 'numpy')
    fp_num = sp.lambdify(x_sym, fp_sym, 'numpy')
    fpp_num = sp.lambdify(x_sym, fpp_sym, 'numpy')
    
    if wahl == 'x · e⁻ˣ':
        xs = np.linspace(-1, 6, 500)
    else:
        xs = np.linspace(-3, 3, 500)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    f_vals = f_num(xs)
    ax.plot(xs, f_vals, 'b-', linewidth=2.5, label=rf'$f(x) = {sp.latex(f_sym)}$')
    
    if zeige_f_prime:
        fp_vals = fp_num(xs)
        ax.plot(xs, fp_vals, 'r--', linewidth=2, label=rf"$f'(x) = {sp.latex(fp_sym)}$")
        # Nullstellen von f' markieren (Extrema)
        for i in range(1, len(xs)):
            if fp_vals[i-1] * fp_vals[i] < 0:
                x_null = xs[i]
                ax.axvline(x=x_null, color='green', linestyle=':', alpha=0.5)
                ax.plot(x_null, f_num(x_null), 'go', markersize=10, zorder=5)
    
    if zeige_f_double_prime:
        fpp_vals = fpp_num(xs)
        ax.plot(xs, fpp_vals, '-', color='orange', linewidth=1.5, alpha=0.8,
                label=rf"$f''(x) = {sp.latex(fpp_sym)}$")
        # Nullstellen von f'' markieren (Wendepunkte)
        for i in range(1, len(xs)):
            if fpp_vals[i-1] * fpp_vals[i] < 0:
                x_null = xs[i]
                ax.axvline(x=x_null, color='orange', linestyle=':', alpha=0.5)
                ax.plot(x_null, f_num(x_null), 'o', color='orange', markersize=8, zorder=5)
    
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ymin, ymax = np.min(f_vals), np.max(f_vals)
    margin = (ymax - ymin) * 0.4
    ax.set_ylim(ymin - margin, ymax + margin)
    ax.legend(fontsize=12, loc='upper right', framealpha=0.9)
    
    # Legende für Punkte
    handles = ax.get_legend().legend_handles
    if zeige_f_prime:
        handles.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=10, label='Extremum'))
    if zeige_f_double_prime:
        handles.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=8, label='Wendepunkt'))
    ax.legend(handles=handles, fontsize=11, loc='upper right', framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

interact(
    zusammenhang,
    wahl=Dropdown(options=list(funktionen.keys()), value='x³ − 3x', description='Funktion:'),
    zeige_f_prime=Checkbox(value=True, description="f'(x) zeigen"),
    zeige_f_double_prime=Checkbox(value=True, description="f''(x) zeigen"),
);

**Beobachte:**
- Grüne Punkte = Extrema (dort ist $f'(x) = 0$)
- Orange Punkte = Wendepunkte (dort ist $f''(x) = 0$)
- Wo $f'(x) > 0$ → $f$ steigt. Wo $f'(x) < 0$ → $f$ fällt.
- Wo $f''(x) > 0$ → linksgekrümmt. Wo $f''(x) < 0$ → rechtsgekrümmt.

---
## 2. Komplette Kurvendiskussion — Schritt für Schritt

Gib eine Funktion ein und lass dir die komplette Kurvendiskussion automatisch berechnen.

In [ ]:
def kurvendiskussion(formel_str='x**3 - 3*x'):
    """Führt eine komplette Kurvendiskussion symbolisch durch."""
    x = sp.Symbol('x')
    f = sp.sympify(formel_str)
    f_prime = sp.diff(f, x)
    f_double_prime = sp.diff(f_prime, x)
    
    display(Markdown(f'### Kurvendiskussion von $f(x) = {sp.latex(f)}$'))
    display(Markdown('---'))
    
    # 1. Definitionsbereich
    display(Markdown(r'**1. Definitionsbereich:** $D = \mathbb{R}$'))
    
    # 2. Symmetrie
    f_neg = f.subs(x, -x)
    if sp.simplify(f_neg - f) == 0:
        sym = 'achsensymmetrisch zur y-Achse ($f(-x) = f(x)$)'
    elif sp.simplify(f_neg + f) == 0:
        sym = 'punktsymmetrisch zum Ursprung ($f(-x) = -f(x)$)'
    else:
        sym = 'keine Symmetrie'
    display(Markdown(f'**2. Symmetrie:** {sym}'))
    
    # 3. Nullstellen
    nullstellen = sp.solve(f, x)
    ns_str = ', '.join([f'$x = {sp.latex(n)}$' for n in nullstellen]) if nullstellen else 'keine'
    display(Markdown(f'**3. Nullstellen** ($f(x) = 0$): {ns_str}'))
    
    # 4. Ableitungen
    display(Markdown(f"**4. Ableitungen:**"))
    display(Markdown(f"$f'(x) = {sp.latex(f_prime)}$"))
    display(Markdown(f"$f''(x) = {sp.latex(f_double_prime)}$"))
    
    # 5. Extremstellen
    extremstellen = sp.solve(f_prime, x)
    display(Markdown(f"**5. Extremstellen** ($f'(x) = 0$):"))
    for xs_val in extremstellen:
        if not xs_val.is_real:
            continue
        y_val = f.subs(x, xs_val)
        fpp_val = f_double_prime.subs(x, xs_val)
        if fpp_val < 0:
            art = 'Hochpunkt (HOP)'
        elif fpp_val > 0:
            art = 'Tiefpunkt (TIP)'
        else:
            art = 'Sattelpunkt (VZW prüfen!)'
        display(Markdown(
            f"- $x = {sp.latex(xs_val)}$: $f({sp.latex(xs_val)}) = {sp.latex(y_val)}$ "
            f"→ $f''({sp.latex(xs_val)}) = {sp.latex(fpp_val)}$ → **{art}**"
        ))
    
    # 6. Wendepunkte
    wendestellen = sp.solve(f_double_prime, x)
    display(Markdown(f"**6. Wendepunkte** ($f''(x) = 0$):"))
    for xs_val in wendestellen:
        if not xs_val.is_real:
            continue
        y_val = f.subs(x, xs_val)
        f3 = sp.diff(f_double_prime, x).subs(x, xs_val)
        bestaetigt = '(bestätigt: $f\'\'\'\\neq 0$)' if f3 != 0 else '(f\'\'\' = 0, genauer prüfen!)'
        display(Markdown(
            f"- $x = {sp.latex(xs_val)}$: $f({sp.latex(xs_val)}) = {sp.latex(y_val)}$ → WP $({sp.latex(xs_val)} | {sp.latex(y_val)})$ {bestaetigt}"
        ))
    
    # 7. Verhalten für x → ±∞
    lim_pos = sp.limit(f, x, sp.oo)
    lim_neg = sp.limit(f, x, -sp.oo)
    display(Markdown(
        f"**7. Verhalten im Unendlichen:** "
        f"$\\lim_{{x \\to +\\infty}} f(x) = {sp.latex(lim_pos)}$, "
        f"$\\lim_{{x \\to -\\infty}} f(x) = {sp.latex(lim_neg)}$"
    ))
    
    # 8. Monotonie
    display(Markdown('**8. Monotonie:**'))
    display(Markdown(f"$f'(x) > 0$ (steigend) bzw. $f'(x) < 0$ (fallend) — siehe Graph oben."))

# Probier es aus! Ändere die Funktion:
kurvendiskussion('x**3 - 3*x')

### Probiere selbst!

| Funktion | Eingabe |
|----------|--------|
| x⁴ − 4x² + 1 | `x**4 - 4*x**2 + 1` |
| −x³ + 6x² − 9x | `-x**3 + 6*x**2 - 9*x` |
| x · eˣ | `x * exp(x)` |
| x² · e⁻ˣ | `x**2 * exp(-x)` |

---
## 3. Graphen zuordnen: f, f' oder f''?

Typische Abituraufgabe! Hier lernst du, Graphen richtig zuzuordnen.

In [ ]:
def zuordnung_quiz():
    """Zeigt 3 Graphen in zufälliger Reihenfolge — welcher ist f, f', f''?"""
    x = np.linspace(-2.5, 2.5, 400)
    
    # f(x) = x³ - 3x
    f = x**3 - 3*x
    fp = 3*x**2 - 3
    fpp = 6*x
    
    graphs = [
        (f, 'Graph A', '#58C4DD'),
        (fp, 'Graph B', '#FC6255'),
        (fpp, 'Graph C', '#FF8C00'),
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    for ax, (vals, name, color) in zip(axes, graphs):
        ax.plot(x, vals, color=color, linewidth=2.5)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.axvline(x=0, color='k', linewidth=0.5)
        ax.grid(True, alpha=0.3)
        ax.set_title(name, fontsize=14, fontweight='bold')
        ax.set_xlim(-2.5, 2.5)
    
    plt.suptitle('Welcher Graph gehört zu f, f\' und f\'\' ?', fontsize=15)
    plt.tight_layout()
    plt.show()
    
    print('Tipps:')
    print('• Wo hat der kubische Graph Extrema? → Dort muss f\' eine Nullstelle haben.')
    print('• f\'\' ist die Ableitung von f\' → welcher Graph ist die Ableitung von welchem?')
    print('• f\'\' ist linear → das kann nur eine Gerade sein.')

zuordnung_quiz()

<details>
<summary><b>Lösung aufklappen</b></summary>

- **Graph A** = $f(x) = x^3 - 3x$ (kubisch, 2 Extrema)
- **Graph B** = $f'(x) = 3x^2 - 3$ (Parabel, Nullstellen bei $x = \pm 1$)
- **Graph C** = $f''(x) = 6x$ (Gerade, Nullstelle bei $x = 0$ = Wendepunkt von f)

</details>

---
## 4. Monotonie und Krümmung sehen

Verschiebe den Punkt und beobachte gleichzeitig **Monotonie** (steigt/fällt) und **Krümmung** (links/rechts).

In [ ]:
from ipywidgets import FloatSlider

def monotonie_kruemmung(x0=0.0):
    f = lambda x: x**3 - 3*x
    fp = lambda x: 3*x**2 - 3
    fpp = lambda x: 6*x
    
    xs = np.linspace(-2.5, 2.5, 300)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(xs, f(xs), color='#58C4DD', linewidth=2.5, label=r'$f(x) = x^3 - 3x$')
    
    # Punkt markieren
    y0 = f(x0)
    ax.plot(x0, y0, 'o', color='yellow', markersize=12, zorder=5)
    
    # Tangente
    m = fp(x0)
    tang = m * (xs - x0) + y0
    ax.plot(xs, tang, '--', color='#FC6255', linewidth=1.5, alpha=0.7)
    
    # Monotonie-Info
    if m > 0.01:
        mono = r'$f\'(x_0) > 0$ → steigend $\nearrow$'
        mono_color = 'green'
    elif m < -0.01:
        mono = r'$f\'(x_0) < 0$ → fallend $\searrow$'
        mono_color = 'red'
    else:
        mono = r'$f\'(x_0) = 0$ → Extremstelle!'
        mono_color = 'yellow'
    
    # Krümmungs-Info
    k = fpp(x0)
    if k > 0.01:
        kruemm = r'$f\'\'(x_0) > 0$ → linksgekrümmt $\smile$'
    elif k < -0.01:
        kruemm = r'$f\'\'(x_0) < 0$ → rechtsgekrümmt $\frown$'
    else:
        kruemm = r'$f\'\'(x_0) = 0$ → Wendepunkt!'
    
    # Infobox
    info = f'$x_0 = {x0:.1f}$\n'
    info += f'$f(x_0) = {y0:.2f}$\n'
    info += f"$f'(x_0) = {m:.2f}$\n"
    info += f"$f''(x_0) = {k:.2f}$"
    ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7),
            color='white', family='monospace')
    
    ax.set_title(f'{mono}     |     {kruemm}', fontsize=13)
    
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-5, 5)
    ax.legend(fontsize=12, loc='lower right', framealpha=0.9)
    plt.tight_layout()
    plt.show()

interact(
    monotonie_kruemmung,
    x0=FloatSlider(value=0.0, min=-2.3, max=2.3, step=0.1, description='$x_0$:',
                   style={'description_width': '30px'}, layout={'width': '500px'})
);

**Aufgabe:** Finde durch Verschieben des Sliders:
1. Die beiden **Extremstellen** (wo $f'(x_0) = 0$)
2. Den **Wendepunkt** (wo $f''(x_0) = 0$)
3. Prüfe: Ist das Extremum bei $x = -1$ ein Maximum oder Minimum? Was sagt $f''(-1)$?